In [22]:
import torch
from torch import nn

In [23]:
def stable_softmax(
    input_logits: torch.Tensor,
    dimension: int,
) -> torch.Tensor:
    """지정한 dimension을 따라 numerically stable softmax를 계산"""
    
    # 가장 큰 Logit을 빼도 지수로 올려서 계산하기 때문에, softmax 결과는 변하지 않는다.
    maximum_logits = input_logits.max(
        dim=dimension,
        keepdim=True,
    ).values
    
    shifted_logits = input_logits - maximum_logits
    exponentials = shifted_logits.exp()
    
    # keepdim=True로 유지해야 원래 Tensor와 broadcasting 할 수 있음.
    normalization = exponentials.sum(
        dim=dimension,
        keepdim=True,
    )

    return exponentials / normalization



# Shape: [B, K, D, H, W] = [1, 4, 1, 1, 1]
tiny_logits = torch.tensor(
    [0.0, 1.0, 2.0, 3.0], # class 별 logits
    dtype=torch.float32,
).reshape(1, 4, 1, 1, 1)

scratch_probabilities = stable_softmax(
    input_logits=tiny_logits,
    dimension=1,
)

# Concise Implementation
reference_probabilities = torch.softmax(
    tiny_logits,
    dim=1,
)

print("Logits:      ", tiny_logits.reshape(-1))
print("Probabilities:", scratch_probabilities.reshape(-1))
print(
    "Probability sum:",
    scratch_probabilities.sum(dim=1).item(),
)

assert torch.allclose(
    scratch_probabilities,
    reference_probabilities,
)

assert torch.allclose(
    scratch_probabilities.sum(dim=1),
    torch.ones(1, 1, 1, 1),
)

Logits:       tensor([0., 1., 2., 3.])
Probabilities: tensor([0.0321, 0.0871, 0.2369, 0.6439])
Probability sum: 1.0


In [24]:
def naive_softmax(
    input_logits: torch.Tensor,
    dimension: int,
) -> torch.Tensor:
    """Maximum subtraction이 없는 불안정한 softmax를 계산한다."""
    
    exponentials = input_logits.exp()
    normalization = exponentials.sum(
        dim=dimension,
        keepdim=True,
    )

    return exponentials / normalization


extreme_logits = torch.tensor(
    [1000.0, 1001.0, 1002.0],
    dtype=torch.float32,
).reshape(1, 3, 1, 1, 1)

naive_probabilities = naive_softmax(
    input_logits=extreme_logits,
    dimension=1,
)

stable_probabilities = stable_softmax(
    input_logits=extreme_logits,
    dimension=1,
)

print("Extreme logits:      ", extreme_logits.reshape(-1))
print("Naive probabilities: ", naive_probabilities.reshape(-1))
print("Stable probabilities:", stable_probabilities.reshape(-1))
print(
    "Stable probability sum:",
    stable_probabilities.sum(dim=1).item(),
)

Extreme logits:       tensor([1000., 1001., 1002.])
Naive probabilities:  tensor([nan, nan, nan])
Stable probabilities: tensor([0.0900, 0.2447, 0.6652])
Stable probability sum: 1.0


In [25]:
# Logits       [B,K,D,H,W]
#                 ↓ softmax
# Probabilities [B,K,D,H,W]

# Target       [B,D,H,W]
#                 ↓ unsqueeze
# Target index [B,1,D,H,W]

# Probabilities + Target index
#                 ↓ gather
#               [B,1,D,H,W]
#                 ↓ squeeze
# Voxel loss    [B,D,H,W]


def voxelwise_negative_log_likelihood(
    class_probabilities: torch.Tensor, # 클래스별 확률 :[B, K, D, H, W]
    target: torch.Tensor,              #  정답 라벨   :[B, D, H, W]
) -> torch.Tensor:
    """각 voxel의 정답 class probability로 NLL을 계산한다."""
    
    # 정답 Target Index: [B, 1, D, H, W]
    target_indices = target.unsqueeze(dim=1)

    # 각 voxel에서 정답 class의 probability만 선택
    # [B, K, D, H, W] -> [B, 1, D, H, W]
    target_probabilities = class_probabilities.gather(
        dim=1,
        index=target_indices,
    )
    
    # [B, 1, D, H, W] -> [B, D, H, W]
    target_probabilities = target_probabilities.squeeze(dim=1)
    
    return -target_probabilities.log()


# Segmentation logits: 
# Shape: [B, K, D, H, W] = [1, 3, 1, 1, 2]
segmentation_logits = torch.tensor(
    [
        [
            [[[2.0, 0.0]]],  # Class 0 logits
            [[[1.0, 1.0]]],  # Class 1 logits
            [[[0.0, 2.0]]],  # Class 2 logits
        ]
    ],
    dtype=torch.float32,
)

# Segmentation target shape:
# Shape: [B, D, H, W] = [1, 1, 1, 2]
segmentation_target = torch.tensor(
    [[[[0, 1]]]], # 첫 voxel의 정답은 class 0, 두 번째 voxel의 정답은 class 1
    dtype=torch.int64,
)

# softmax로 각각의 logits를 확률화
# Shape: [B, K, D, H, W] = [1, 3, 1, 1, 2]
segmentation_probabilities = stable_softmax(
    input_logits=segmentation_logits,
    dimension=1,
)

# Shape: [B, D, H, W] = [1, 1, 1, 2]
voxel_losses = voxelwise_negative_log_likelihood(
    class_probabilities=segmentation_probabilities,
    target=segmentation_target,
)

mean_loss = voxel_losses.mean()

print(segmentation_probabilities.shape)
print(voxel_losses.shape)

print(
    "Class probabilities:\n",
    segmentation_probabilities[:, :, 0, 0, :],
)
print("Target class IDs:", segmentation_target.reshape(-1))
print("Voxel NLL:", voxel_losses.reshape(-1))
print("Mean NLL:", mean_loss.item())

torch.Size([1, 3, 1, 1, 2])
torch.Size([1, 1, 1, 2])
Class probabilities:
 tensor([[[0.6652, 0.0900],
         [0.2447, 0.2447],
         [0.0900, 0.6652]]])
Target class IDs: tensor([0, 1])
Voxel NLL: tensor([0.4076, 1.4076])
Mean NLL: 0.9076059460639954


In [ ]:
def scratch_cross_entropy(
    input_logits: torch.Tensor,
    target: torch.Tensor,
) -> torch.Tensor:
    """Softmax와 voxel-wise NLL로 mean Cross-Entropy를 계산"""
    
    # logits/softmax: [B, K, D, H, W]
    class_probabilities = stable_softmax(
        input_logits=input_logits,
        dimension=1,
    )
    
    # 각 voxel마다 loss 하나가 나옴
    # : [B, D, H, W]
    voxel_losses = voxelwise_negative_log_likelihood(
        class_probabilities=class_probabilities,
        target=target,
    )
    
    return voxel_losses.mean()



# Gradient를 저장할 수 있도록 leaf Tensor를 새로 만듬.
# Gradient shape: [B, K, D, H, W]
# 각 logit이 loss에 얼마나 영향을 주었는지 계산하므로 gradient shape는 logits shape와 동일
trainable_logits = (
    segmentation_logits
    .clone()
    .detach()
    .requires_grad_(True)
)

scratch_loss = scratch_cross_entropy(
    input_logits=trainable_logits,
    target=segmentation_target,
)

cross_entropy_loss = nn.CrossEntropyLoss(
    reduction="mean",
)

# PyTorch CrossEntropyLoss에는 raw logits를 그대로 전달
# 내부적으로 numerically stable log_softmax + NLLLoss를 수행
reference_loss = cross_entropy_loss(
    trainable_logits,
    segmentation_target,
)

print("Scratch loss:  ", scratch_loss.item())
print("PyTorch loss:  ", reference_loss.item())
print(
    "Absolute difference:",
    (scratch_loss - reference_loss).abs().item(),
)

# Scalar loss에서 각 voxel/class logit까지 gradient를 전파한다.
reference_loss.backward()


print("Gradient shape:", trainable_logits.grad.shape)
print("Gradient:\n", trainable_logits.grad[:, :, 0, 0, :])

Scratch loss:   0.9076059460639954
PyTorch loss:   0.9076058864593506
Absolute difference: 5.960464477539063e-08
Gradient shape: torch.Size([1, 3, 1, 1, 2])
Gradient:
 tensor([[[-0.1674,  0.0450],
         [ 0.1224, -0.3776],
         [ 0.0450,  0.3326]]])
